# The k-means algorithm 
In this exercise we will build a k-means model that will assign iris flowers to one of three species based on their measurements.

We will build on the prepared data from the last Jupyter notebook.

## Loading and splitting the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
my_arrays = np.load("iris_numpy.npz")
X = my_arrays['X']
y = my_arrays['y']
X_norm = my_arrays['X_norm']
X_features = my_arrays['X_feature']

Splitting the data into training and test sets

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=42)

## Determining the number of k segments
The input parameter for k-means is the number of segments the data should be split into.

If the number of segments is part of the assignment, as it is for us with the irises, then there's nothing to figure out.

But we will show a procedure for determining the optimal number of segments from the data, in case it isn't clear how many parts the data should be split into.

We will run the k-means algorithm several times with different numbers of segments. For each resulting model we will track the value of **inertia**.

Inertia is the sum of squared distances of samples to their closest cluster center, weighted by the sample weights if provided.

In [ ]:
from sklearn.cluster import KMeans
from scipy.stats import mode

inertia_list = []
for num_clusters in range(1, 10):
    kmeans_model = KMeans(n_clusters=num_clusters, init="k-means++", n_init = 10)
    kmeans_model.fit(X_norm)
    inertia_list.append(kmeans_model.inertia_)

The optimal number of segments is where the inertia value changes significantly between two models whose number of segments differs by one.

We can find this, for example, by plotting inertia on an elbow chart. The optimal k is where the chart last bends significantly.

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(range(1,10),inertia_list)
plt.scatter(range(1,10),inertia_list)
plt.scatter(3, inertia_list[2], marker="X", s=300, c="r")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia Value")
plt.title("Different Inertia Values for Different Number of Clusters")

## Training K-means 
Creating and training the k-means model.

Notice that the k-means algorithm is an unsupervised model. It does not need to know the correct answers during training.

In [ ]:
kmean_model = KMeans(n_clusters=3, random_state=2, n_init=10)
kmean_model.fit(X_train)

Displaying the coordinates of the cluster centers

In [ ]:
kmean_model.cluster_centers_

## Model prediction
Running the model on the training and test data.

In [ ]:
y_pred_train = kmean_model.predict(X_train)
y_pred_test = kmean_model.predict(X_test)

In [ ]:
print (y_pred_train)

The K-means implementation chooses the cluster ids randomly. This numbering may not correspond to the iris species value as chosen by the label encoder. This is because k-means is an algorithm for creating clusters and not directly a classification algorithm, which requires having labeled results in the dataset in advance.

To determine the cluster accuracy, we will need to align the cluster ids returned by k-means with the cluster ids chosen by label_encoder.

In the variable y_pred_train we have the k-means cluster numbers.

In the array y_train we have the correct answers - encoded according to the encoder.

In [ ]:
y_train

We will create the mapping as follows.

We go through all three clusters (0, 1, 2).

- mask = (y_train_pred == cluster) creates a boolean mask (True/False) that tells us which samples from the training data belong to the cluster currently being processed.
- y_train[mask] selects the actual (correct) classes of only those samples that k-means assigned to the current cluster.
- mode(y_train[mask], keepdims=True).mode[0] finds the most frequent class (mode) among those samples.
- labels_map[cluster] the most frequent class is stored in the dictionary for mapping.

In [ ]:
labels_map = {}
for cluster in range(3):
    mask = (y_pred_train == cluster)
    labels_map[cluster] = mode(y_train[mask], keepdims=True).mode[0]

Mapping of cluster id to class id

In [ ]:
labels_map

We will load the encoder and display the mapping of class names to numbers.

In [ ]:
import joblib
encoder=joblib.load('classification_encoder.bin')
encoder_mapping = dict(enumerate(encoder.classes_))
encoder_mapping

The complete mapping of cluster id to group name.

In [ ]:
for id in range (0,3):
    print (f"Cluster id {id}, encoding id: {labels_map[id]}, label: {encoder_mapping[labels_map[id]]}")

Remapping the result from the cluster id to the correct label_encoder answers.

In [ ]:
y_pred_train = np.vectorize(labels_map.get)(y_pred_train)
y_pred_test = np.vectorize(labels_map.get)(y_pred_test)

In [ ]:
y_pred_test

## Model visualization
We will create two charts. One will show the predictions, the other the actual values.

Charts for the training data

In [ ]:
plt.figure(figsize=(16,6))

# prediction
plt.subplot(1,2,1)
plt.scatter(X_train[y_pred_train == labels_map[0], 0], X_train[y_pred_train == labels_map[0], 1], s = 50, c = 'purple', label = 'Iris-setosa')
plt.scatter(X_train[y_pred_train == labels_map[1], 0], X_train[y_pred_train == labels_map[1], 1], s = 50, c = 'orange', label = 'Iris-versicolour')
plt.scatter(X_train[y_pred_train == labels_map[2], 0], X_train[y_pred_train == labels_map[2], 1], s = 50, c = 'green', label = 'Iris-virginica')
plt.title('Predicted Species'); plt.xlabel('petal_length'); plt.ylabel('petal_width')
# centroid
plt.scatter(kmean_model.cluster_centers_[:, 0], kmean_model.cluster_centers_[:,1], s = 100, c = 'red', label = 'Centroids')
plt.legend()

# real values
plt.subplot(1,2,2)
plt.scatter(X_train[y_train == labels_map[0], 0], X_train[y_train == labels_map[0], 1], s = 50, c = 'purple', label = 'Iris-setosa')
plt.scatter(X_train[y_train == labels_map[1], 0], X_train[y_train == labels_map[1], 1], s = 50, c = 'orange', label = 'Iris-versicolour')
plt.scatter(X_train[y_train == labels_map[2], 0], X_train[y_train == labels_map[2], 1], s = 50, c = 'green', label = 'Iris-virginica')
plt.title('True Species'); plt.xlabel('petal_length'); plt.ylabel('petal_width')

plt.legend()

Charts for the test data

In [ ]:
plt.figure(figsize=(16,6))

# prediction
plt.subplot(1,2,1)
plt.scatter(X_test[y_pred_test == labels_map[0], 0], X_test[y_pred_test == labels_map[0], 1], s = 50, c = 'purple', label = 'Iris-setosa')
plt.scatter(X_test[y_pred_test == labels_map[1], 0], X_test[y_pred_test == labels_map[1], 1], s = 50, c = 'orange', label = 'Iris-versicolour')
plt.scatter(X_test[y_pred_test == labels_map[2], 0], X_test[y_pred_test == labels_map[2], 1], s = 50, c = 'green', label = 'Iris-virginica')
plt.title('Predicted Species'); plt.xlabel('petal_length'); plt.ylabel('petal_width')
# centroid
plt.scatter(kmean_model.cluster_centers_[:, 0], kmean_model.cluster_centers_[:,1], s = 100, c = 'red', label = 'Centroids')
plt.legend()

# real values
plt.subplot(1,2,2)
plt.scatter(X_test[y_test == labels_map[0], 0], X_test[y_test == labels_map[0], 1], s = 50, c = 'purple', label = 'Iris-setosa')
plt.scatter(X_test[y_test == labels_map[1], 0], X_test[y_test == labels_map[1], 1], s = 50, c = 'orange', label = 'Iris-versicolour')
plt.scatter(X_test[y_test == labels_map[2], 0], X_test[y_test == labels_map[2], 1], s = 50, c = 'green', label = 'Iris-virginica')
plt.title('True Species'); plt.xlabel('petal_length'); plt.ylabel('petal_width')
plt.legend()

## Model evaluation
We can evaluate how well the model works using various metrics. The task's requirements will suggest which metric to choose.

Metrics for evaluating a classification model:

| Metric | Formula | Description |
|---------|--------|-------|
| **Accuracy** | (TP + TN) / (TP + TN + FP + FN) | Proportion of correct answers |
| **Recall** (Sensitivity, TPR) | TP / (TP + FN) | Of those that are positive, how many did we correctly catch |
| **Specificity** (TNR) | TN / (TN + FP) | Of those that are negative, how many did we correctly reject |
| **Precision** | TP / (TP + FP) | Of those we labeled as positive, how many actually are |
| **F1** | 2 × (Precision × Recall) / (Precision + Recall) | Harmonic mean of precision and recall |

Abbreviations: TP = True Positive, TN = True Negative, FP = False Positive, FN = False Negative

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

Predictions for the test data, which we have already calculated from the visualization.

In [ ]:
y_pred_test

The confusion matrix shows how the model predicts for each class. The correct predictions are on the main diagonal. The other cells show the mix-ups.

In [ ]:
cf_matrix=confusion_matrix(y_test, y_pred_test)
sns.heatmap(cf_matrix, annot=True)

The model's score

In [ ]:
score=accuracy_score(y_test, y_pred_test)
print (score)

## Saving the model
The model can again be saved to a file for use in inference.

In [ ]:
filename = 'kmean_model.bin'
joblib.dump(kmean_model, filename)

# Using the model

We will try to use the model to determine 3 different irises. The input values are given in the original units, the same as the original data.

In [ ]:
iris_measurements = np.array([[5.1, 1.9],[1.0, 0.5],[6.0, 2.5]])

The model was trained on normalized data, so we need to normalize the input data.

In [ ]:
scaler=joblib.load('classification_std_scaler.bin')

In [ ]:
iris_measurements_norm=scaler.transform(iris_measurements)
iris_measurements_norm

Displaying the encoder and its mapping of classes to numbers.

In [ ]:
encoder=joblib.load('classification_encoder.bin')
encoder_mapping = dict(enumerate(encoder.classes_))
encoder_mapping

We will load the saved model.

In [ ]:
loaded_model = joblib.load(filename)

We run the model with the predict function. The model's output is an array of numbers identifying the given species.

In [ ]:
predictions = loaded_model.predict(iris_measurements_norm)
predictions

We convert the numeric answers into a readable form.

In [ ]:
for prediction in predictions:
    print (f"Cluster id {prediction}, encoding id: {labels_map[prediction]}, label: {encoder_mapping[labels_map[prediction]]}")

## Hyperparameter tuning
The k-means algorithm has various parameters. One of them is the way distance is measured.

The following procedure will print out the most suitable combination of parameters for the given data.

It simply creates models for various combinations of parameters and measures their accuracy.

In [ ]:
from sklearn.model_selection import GridSearchCV

select_params={
               'algorithm' :["lloyd", "elkan"],
              }

grid_kmean = GridSearchCV(kmean_model, select_params, cv=5)
grid_kmean.fit(X_norm)

print('Best parameters: {}'.format(grid_kmean.best_params_))
print('Best score on training set: {}'.format(grid_kmean.best_score_))

## Building a model with a pre-prepared variable

We will build a new model that uses only one artificially created variable, according to the formula petal_length * petal_width.

Splitting the data into training and test sets. The library expects X to have multiple variables, so we need to use reshape.

In [ ]:
from sklearn.model_selection import train_test_split
X_feature_train, X_feature_test, y_feature_train, y_feature_test = train_test_split(X_features.reshape(-1,1), y, test_size=0.2, random_state=42)

In [ ]:
kmean_feature_model = KMeans(n_clusters=3, random_state= 2, n_init=10)
kmean_feature_model.fit(X_feature_train)

A new model, a new mapping of cluster ids to labels.

In [ ]:
y_feature_pred_train = kmean_feature_model.predict(X_feature_train)
y_feature_pred_test = kmean_feature_model.predict(X_feature_test)

labels_feature_map = {}
for cluster in range(3):
    mask = (y_feature_pred_train == cluster)
    labels_feature_map[cluster] = mode(y_feature_train[mask], keepdims=True).mode[0]

In [ ]:
for id in range (0,3):
    print (f"Cluster id {id}, encoding id: {labels_feature_map[id]}, label: {encoder_mapping[labels_feature_map[id]]}")

Remapping the result from the cluster id to the correct label_encoder answers.

In [ ]:
y_feature_pred_train = np.vectorize(labels_feature_map.get)(y_feature_pred_train)
y_feature_pred_test = np.vectorize(labels_feature_map.get)(y_feature_pred_test)

In [ ]:
cf_matrix=confusion_matrix(y_feature_test, y_feature_pred_test)
sns.heatmap(cf_matrix, annot=True)

The score is somewhat lower than for the model with two variables.

In [ ]:
accuracy_score(y_feature_test, y_feature_pred_test)

Displaying the model's result compared against the actual values.

In [ ]:
X_plot = X_feature_test
y_true = y_feature_test
y_pred = y_feature_pred_test

plt.figure(figsize=(8, 4))
plt.scatter(X_plot, np.zeros_like(X_plot), c=y_true, cmap='viridis', s=100, marker='o', label='Real classes')
plt.scatter(X_plot, np.ones_like(X_plot)*0.1, c=y_pred, cmap='viridis', s=100, marker='x', label='Predicted clusters')
plt.yticks([])
plt.xlabel('Petal area (length × width)')
plt.legend()
plt.title('Comparison of actual classes and predicted clusters')
plt.show()